In [2]:
import pandas as pd

# 1. Load the data using your paths
df_roads = pd.read_csv("../data_cleaned_by_lecturer/_roads3.csv", low_memory=False)
df_bmms = pd.read_excel("../data_cleaned_by_lecturer/BMMS_overview.xlsx")

# 2. Filter to just the N1 road
df_roads_n1 = df_roads[df_roads['road'] == 'N1'].copy()
df_bmms_n1 = df_bmms[df_bmms['road'] == 'N1'].copy()

# 3. Filter df_roads to only look at components categorized as bridges/culverts
# (Using your map_to_model_type logic)
bridge_mask = df_roads_n1['type'].astype(str).str.lower().str.contains('bridge|culvert', na=False)
df_roads_bridges = df_roads_n1[bridge_mask].copy()

# 4. Check the raw overlap
raw_roads_lrps = set(df_roads_bridges['lrp'].dropna())
raw_bmms_lrps = set(df_bmms_n1['LRPName'].dropna())

raw_matches = raw_roads_lrps.intersection(raw_bmms_lrps)

print(f"Total bridges/culverts in N1 roads file: {len(df_roads_bridges)}")
print(f"Exact LRP matches BEFORE cleaning: {len(raw_matches)}")

Total bridges/culverts in N1 roads file: 798
Exact LRP matches BEFORE cleaning: 382


In [3]:
# 1. Clean the LRP strings: remove all non-alphanumeric characters and make uppercase
df_roads_bridges['lrp_clean'] = df_roads_bridges['lrp'].astype(str).str.replace(r'\W+', '', regex=True).str.upper()
df_bmms_n1['LRPName_clean'] = df_bmms_n1['LRPName'].astype(str).str.replace(r'\W+', '', regex=True).str.upper()

# 2. Check the new overlap
clean_roads_lrps = set(df_roads_bridges['lrp_clean'].dropna())
clean_bmms_lrps = set(df_bmms_n1['LRPName_clean'].dropna())

clean_matches = clean_roads_lrps.intersection(clean_bmms_lrps)

print(f"Exact LRP matches AFTER cleaning: {len(clean_matches)}")
print(f"Bridges recovered: {len(clean_matches) - len(raw_matches)}")

Exact LRP matches AFTER cleaning: 382
Bridges recovered: 0


In [4]:
# Find LRPs that are in the roads dataset but STILL not in the BMMS dataset
unmatched_lrps = clean_roads_lrps - clean_bmms_lrps

print(f"There are {len(unmatched_lrps)} bridges still missing length data from BMMS.")
print("\nHere are a few of the stubborn LRPs from the roads file:")

# Display the original rows for a few of these unmatched LRPs
stubborn_df = df_roads_bridges[df_roads_bridges['lrp_clean'].isin(unmatched_lrps)]
display(stubborn_df[['road', 'chainage', 'lrp', 'lrp_clean', 'type']].head(15))

There are 416 bridges still missing length data from BMMS.

Here are a few of the stubborn LRPs from the roads file:


,road,chainage,lrp,lrp_clean,type
1,N1,0.814,LRPSa,LRPSA,Culvert
5,N1,2.130,LRP002a,LRP002A,Culvert
14,N1,8.011,LRP008a,LRP008A,Bridge
24,N1,11.313,LRP011b,LRP011B,Culvert
33,N1,14.563,LRP015a,LRP015A,Culvert
36,N1,16.242,LRP016b,LRP016B,Bridge
37,N1,16.402,LRP016c,LRP016C,Bridge
39,N1,16.831,LRP017a,LRP017A,Bridge
41,N1,17.204,LRP017c,LRP017C,Bridge
42,N1,17.234,LRP017d,LRP017D,Bridge
